# Data exploration
Run `python download_data.py` first, then run the cells top to bottom.

In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter
from src.preprocess import add_features, preprocess_text

df = pd.read_csv(ROOT / "data" / "clean.csv")
print(df.shape)
df.head()

## 1. Class balance

In [ ]:
print(df.groupby(["source", "label"]).size().unstack(fill_value=0))
df["label"].value_counts(normalize=True).rename({0: "safe", 1: "phishing"}).plot.bar(title="Share of each class")
plt.show()

## 2. Text length

In [ ]:
df["n_words"] = df["text"].str.split().str.len()
print(df.groupby("label")["n_words"].describe())
df["n_words"].clip(upper=1000).plot.hist(bins=50, title="Words per message (clipped at 1000)")
plt.show()

## 3. What does preprocessing do?

In [ ]:
for raw in df.sample(4, random_state=42)["text"]:
    print("RAW   :", raw[:150].replace("\n", " "))
    print("TOKENS:", preprocess_text(raw))
    print()

## 4. Is `has_url` informative?

In [ ]:
feat = add_features(df.sample(min(len(df), 5000), random_state=42))
print(feat.groupby("has_url")["label"].agg(["mean", "count"]).rename(columns={"mean": "phishing_rate"}))

## Notes
- How imbalanced is the data? Does it differ by `source`?
- Which `max_len` covers most messages?
- Any words that give away the label too easily?

In [ ]:
for label, name in [(1, "phishing"), (0, "safe")]:
    counts = Counter(t for toks in feat.loc[feat.label == label, "tokens"] for t in toks)
    print(name, counts.most_common(15))

## Takeaways to write down
- How imbalanced is the data? Does it differ by `source` (URLs vs emails)?
- Which `max_len` covers most messages?
- Are there words that leak the label trivially (e.g. dataset artefacts)? If yes, treat the scores with suspicion.